# Notebook 2: Data Cleaning

**Automotive Car Price Prediction Pipeline**

---

This notebook cleans the raw dataset:
- Checks and reports **missing values** (no imputation — done after split in Notebook 5)
- Removes **corrupt/placeholder data** (rule-based, not statistical)
- Checks and reports **outliers** (no removal)
- Renames columns and standardizes text values

**Input:** `workspace.default.cars_new` (from Notebook 1)

**Output:** `workspace.default.cleaned_data`

## Load Data from Catalog

In [0]:
import pandas as pd
import numpy as np

# Read from catalog into pandas
df = spark.table("workspace.default.cars_new").toPandas()
initial_count = len(df)
print(f"Loaded raw_data: ({initial_count} rows, {len(df.columns)} columns)")

Loaded raw_data: (56244 rows, 12 columns)


## 1. Datatype Conversion

In [0]:
# Convert volume_c3 from object to integer
df['volume_cm3'] = pd.to_numeric(df['volume_cm3'], errors='coerce').astype('Int64')

## 2. Missing Values — Check and Report Only

> **Note:** We do NOT impute here. Imputation will be done **after the train/test split** in Notebook 5 to avoid data leakage.

In [0]:
df.isnull().sum()

make                     0
model                    0
priceUSD                 0
year                     0
condition                0
mileage_kilometers       0
fuel_type                0
volume_cm3              47
color                    0
transmission             0
drive_unit            1905
segment               5291
dtype: int64

In [0]:
# Investigate all rows where volume_cm3 is null #impute with zero
null_volume = df[df['volume_cm3'].isnull()]
display(null_volume)


make,model,priceUSD,year,condition,mileage_kilometers,fuel_type,volume_cm3,color,transmission,drive_unit,segment
eksklyuziv,redkaya-model,2700,2001,with mileage,111.0,electrocar,null,green,auto,front-wheel drive,null
eksklyuziv,sport,6500,2015,with mileage,10.0,electrocar,null,green,auto,null,null
chevrolet,volt,22000,2015,with mileage,157715.32,electrocar,null,silver,auto,front-wheel drive,null
cadillac,elr,25900,2014,with mileage,70000.0,electrocar,null,silver,auto,front-wheel drive,null
volkswagen,passat,9200,2011,with mileage,185000.0,electrocar,null,silver,auto,front-wheel drive,D
volkswagen,passat,7499,2007,with mileage,530000.0,electrocar,null,silver,mechanics,front-wheel drive,D
bmw,i3,38000,2018,with mileage,67000.0,electrocar,null,other,auto,rear drive,null
bmw,i3,57383,2017,with mileage,2822.0,electrocar,null,blue,auto,rear drive,null
bmw,i3,22900,2014,with mileage,71500.0,electrocar,null,silver,auto,rear drive,null
bmw,i3,22900,2015,with mileage,50000.0,electrocar,null,black,auto,rear drive,null


## 3. Remove Corrupt / Placeholder Data (Rule-Based)

- These removals use hardcoded rules, not data statistics — safe to do before the split.
- Statistical outlier removal happens after the split in Notebook 5.

In [0]:
before = len(df)

# Remove placeholder mileage values (9,999,999 km) — clearly corrupt data
n1 = (df['mileage_kilometers'] >= 9_999_999).sum()
df = df[df['mileage_kilometers'] < 9_999_999]
print(f"Removed {n1} rows with mileage = 9,999,999 (placeholder values)")

# Remove very low prices (< $100) — clearly corrupt data
n2 = (df['priceUSD'] < 100).sum()
df = df[df['priceUSD'] >= 100]
print(f"Removed {n2} rows with price < $100 (data errors)")

after = len(df)
print(f"\nTotal rows removed: {before - after}")
print(f"Rows remaining: {after}")

Removed 22 rows with mileage = 9,999,999 (placeholder values)
Removed 2 rows with price < $100 (data errors)

Total rows removed: 24
Rows remaining: 56220


## 4. Drop Duplicated Rows

In [0]:
# Check for duplicated rows
n_duplicates = df.duplicated().sum()
print(f"Duplicated rows found: {n_duplicates}")

# Drop duplicated rows
df = df.drop_duplicates()
print(f"Rows after dropping duplicates: {len(df)}")

Duplicated rows found: 87
Rows after dropping duplicates: 56133


## 5. Outlier Check — Report Only

> **Note:** We only report outliers here. 

In [0]:
# Outlier report — IQR method on numeric columns
print("Outlier Report (IQR method — 1.5 * IQR rule):")
print("=" * 60)

numeric_cols_check = df.select_dtypes(include=[np.number]).columns.tolist()
outlier_summary = []

for col in numeric_cols_check:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = n_outliers / len(df) * 100
    outlier_summary.append((col, n_outliers, pct, lower, upper))
    if n_outliers > 0:
        print(f"  {col:25s}: {n_outliers:>5} outliers ({pct:.1f}%) | bounds: [{lower:.1f}, {upper:.1f}]")

total_outliers = sum(x[1] for x in outlier_summary)
print(f"\nTotal outlier rows (may overlap): {total_outliers}")


Outlier Report (IQR method — 1.5 * IQR rule):
  priceUSD                 :  2457 outliers (4.4%) | bounds: [-8900.0, 21100.0]
  year                     :   257 outliers (0.5%) | bounds: [1980.0, 2028.0]
  mileage_kilometers       :   665 outliers (1.2%) | bounds: [-122500.0, 569500.0]
  volume_cm3               :  3615 outliers (6.4%) | bounds: [550.0, 3350.0]

Total outlier rows (may overlap): 6994


## 6. Standardize Text Values

In [0]:
# Lowercase all string columns for consistency
string_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"String columns to lowercase: {string_cols}")

for col_name in string_cols:
    df[col_name] = df[col_name].str.lower()

df[string_cols].head()

String columns to lowercase: ['make', 'model', 'condition', 'fuel_type', 'color', 'transmission', 'drive_unit', 'segment']


,make,model,condition,fuel_type,color,transmission,drive_unit,segment
0,nissan,qashqai,with mileage,petrol,white,auto,front-wheel drive,j
1,nissan,qashqai,with mileage,petrol,red,auto,front-wheel drive,j
2,nissan,qashqai,with mileage,petrol,brown,mechanics,front-wheel drive,j
3,nissan,qashqai,with mileage,petrol,black,mechanics,front-wheel drive,j
4,nissan,qashqai,with mileage,petrol,other,auto,part-time four-wheel drive,j


## 7. Final Verification

In [0]:
print("=" * 55)
print("CLEANING SUMMARY")
print("=" * 55)
print(f"Original rows: {initial_count}")
print(f"Final rows:    {len(df)}")
print(f"Rows removed:  {initial_count - len(df)}")
print(f"Columns:       {len(df.columns)}")
print(f"\nRemaining null values (will be imputed in Notebook 5):")
null_remaining = df.isnull().sum()
null_remaining = null_remaining[null_remaining > 0]
if len(null_remaining) == 0:
    print("  None")
else:
    for col_name, cnt in null_remaining.items():
        pct = cnt / len(df) * 100
        print(f"  {col_name:25s}: {cnt} ({pct:.1f}%)")
print(f"\nData types:")
print(df.dtypes.to_string())

CLEANING SUMMARY
Original rows: 56244
Final rows:    56133
Rows removed:  111
Columns:       12

Remaining null values (will be imputed in Notebook 5):
  volume_cm3               : 47 (0.1%)
  drive_unit               : 1904 (3.4%)
  segment                  : 5281 (9.4%)

Data types:
make                   object
model                  object
priceUSD                int64
year                    int64
condition              object
mileage_kilometers    float64
fuel_type              object
volume_cm3              Int64
color                  object
transmission           object
drive_unit             object
segment                object


In [0]:
df.head()

,make,model,priceUSD,year,condition,mileage_kilometers,fuel_type,volume_cm3,color,transmission,drive_unit,segment
0,nissan,qashqai,16800,2016,with mileage,44000.0,petrol,2000,white,auto,front-wheel drive,j
1,nissan,qashqai,16900,2016,with mileage,80000.0,petrol,1200,red,auto,front-wheel drive,j
2,nissan,qashqai,9650,2010,with mileage,198000.0,petrol,1600,brown,mechanics,front-wheel drive,j
3,nissan,qashqai,9800,2011,with mileage,117000.0,petrol,1600,black,mechanics,front-wheel drive,j
4,nissan,qashqai,11500,2011,with mileage,93000.0,petrol,2000,other,auto,part-time four-wheel drive,j


## 8. Save to Catalog

In [0]:
# Convert to Spark and save as Delta table
df_spark = spark.createDataFrame(df)
df_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.cleaned_data")


# Verify
verify = spark.table("workspace.default.cleaned_data").toPandas()
print(f"Saved cleaned_data: ({len(verify)} rows, {len(verify.columns)} columns)")
print(f"Table location: workspace.default.cleaned_data")

Saved cleaned_data: (56133 rows, 12 columns)
Table location: workspace.default.cleaned_data
